# 0-ablations

This notebook is a diagnostic ablation notebook. It does not compare codecs. Its purpose is to measure whether the checkpoint contains the statistical structure that motivates a tensor-aware method:

1. exponent streams have low zero-order empirical entropy;
2. tensors with the same role across blocks have more similar exponent distributions than unrelated tensors.

In [1]:
from neuralzip import HFModel
from neuralzip.ablations import (
    exponent_entropy_by_signature,
    heterogeneity_table,
    transformer_weight_subset,
)

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Model Slice

The default slice uses GPT-2 FP32 and the first 16 transformer weight matrices. The slice keeps repeated tensor roles across blocks while avoiding the token embedding matrix, which would dominate the analysis.

In [2]:
hf_model = HFModel("gpt2", dtype="float32", local_files_only=True)
model_slice = transformer_weight_subset(hf_model, limit=16)

raw_bytes = sum(tensor.numel() * tensor.element_size() for tensor in model_slice.values())
print(f"selected tensors: {len(model_slice)}")
print(f"raw bytes: {raw_bytes:,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

selected tensors: 16
raw bytes: 113,246,208


## Zero-Order Entropy

For each tensor, the exponent field is viewed as a byte-valued stream and measured with zero-order empirical entropy. The table aggregates by tensor role rather than by layer name.

In [3]:
entropy_summary = exponent_entropy_by_signature(model_slice)
entropy_summary

,signature,tensors,total_exponents,mean_h0_bits,median_h0_bits,min_h0_bits,max_h0_bits,mean_dominant_exponent_mass
3,h.#.mlp.c_proj.weight,4,9437184,2.631465,2.621533,2.612141,2.670655,0.301957
2,h.#.mlp.c_fc.weight,4,9437184,2.636422,2.622187,2.579550,2.721765,0.283064
0,h.#.attn.c_attn.weight,4,7077888,2.770896,2.758643,2.645909,2.920390,0.255553
1,h.#.attn.c_proj.weight,4,2359296,2.774579,2.646947,2.613926,3.190495,0.273892


In [4]:
overall_h0 = (
    entropy_summary["mean_h0_bits"]
    .mul(entropy_summary["total_exponents"])
    .sum()
    / entropy_summary["total_exponents"].sum()
)

print(f"weighted mean exponent H0: {overall_h0:.4f} bits/symbol")
print("raw exponent width: 8.0000 bits/symbol")
print(f"mean dominant exponent mass: {entropy_summary['mean_dominant_exponent_mass'].mean():.4f}")

weighted mean exponent H0: 2.6799 bits/symbol
raw exponent width: 8.0000 bits/symbol
mean dominant exponent mass: 0.2786


## Blockwise Heterogeneity

The same exponent histograms are compared with Jensen-Shannon distance. Same-signature pairs correspond to the same tensor role across different transformer blocks, for example all attention projection weights. Lower same-signature distances indicate that layer role is a meaningful statistical partition.

In [5]:
hist_table, js_pairs, heterogeneity_summary = heterogeneity_table(model_slice)
heterogeneity_summary

,same_signature,pairs,mean_js,median_js,min_js,max_js
1,True,24,0.085357,0.046154,0.006067,0.322560
0,False,96,0.196840,0.216799,0.009770,0.384786


In [6]:
js_quantiles = (
    js_pairs
    .groupby("same_signature", as_index=False)
    .agg(
        pairs=("js_distance", "size"),
        q10_js=("js_distance", lambda x: x.quantile(0.10)),
        q25_js=("js_distance", lambda x: x.quantile(0.25)),
        median_js=("js_distance", "median"),
        q75_js=("js_distance", lambda x: x.quantile(0.75)),
        q90_js=("js_distance", lambda x: x.quantile(0.90)),
    )
    .sort_values("same_signature", ascending=False)
)
js_quantiles

,same_signature,pairs,q10_js,q25_js,median_js,q75_js,q90_js
1,True,24,0.022734,0.029064,0.046154,0.081621,0.265783
0,False,96,0.036636,0.123776,0.216799,0.252030,0.316717


In [7]:
same = js_pairs[js_pairs["same_signature"]]["js_distance"]
different = js_pairs[~js_pairs["same_signature"]]["js_distance"]

print(f"same-signature median JS: {same.median():.4f}")
print(f"different-signature median JS: {different.median():.4f}")
print(f"median gap: {different.median() - same.median():.4f}")

same-signature median JS: 0.0462
different-signature median JS: 0.2168
median gap: 0.1706
